<a href="https://colab.research.google.com/github/MyNameisMick/MyNameisMick/blob/main/Go(FinishVersion).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [170]:
import matplotlib.pyplot as plt
import random
import ipywidgets as widgets
from IPython.display import display, clear_output


# ตั้งค่าเก็บตัวแปร
#ขนาดกระดาน
SIZE = 19

# 0 = มีช่องว่าง, 1 = หมากดำ, 2 = หมากขาว
board = [[0] * SIZE for _ in range(SIZE)]

# เริ่มแรกจะให้คนเป็นดำ, บอทเป็นขาว (แต่เปลี่ยนได้)
human_player = 1
bot_player = 2

# ตัวแปรผู้เล่น
current_player = 1

# ตัวแปรคะแนนเสริม (5.5 คะแนนตามสากล), เก็บ bool ไว้ true ตอนจบเกม
KOMI = 5.5
game_over = False

# ตัวแปรเก็บหมากที่ถูกล้อมได้ในเกม
black_captures = 0
white_captures = 0

# ตัวแปรจำกัดหมาก แล้วก็เก็บการใช้หมาก
stone_limit = 181
black_used_stones = 0
white_used_stones = 0

# ตัวแปรเก็บจำนวนครั้งที่ผ่าน
consecutive_passes = 0


# ฟังก์ชันพื้นฐาน เช่น Reset game,ขนาดขนาดของหมาก, การสลับฝั่ง และชื่อ
def reset_game(size=19, human=1):
    global SIZE, board, human_player, bot_player, current_player
    global black_captures, white_captures, game_over
    global stone_limit, black_used_stones, white_used_stones
    global consecutive_passes

    SIZE = size
    board = [[0] * SIZE for _ in range(SIZE)]
    human_player = human
    bot_player = 2 if human == 1 else 1
    current_player = 1

    black_captures = 0
    white_captures = 0
    game_over = False

    stone_limit = get_stone_limit(SIZE)
    black_used_stones = 0
    white_used_stones = 0

    consecutive_passes = 0

    print(f"เริ่มเกมใหม่: กระดาน {SIZE}x{SIZE}")
    print(f"ผู้เล่น = {'หมากดำ' if human_player == 1 else 'หมากขาว'}")
    print(f"ศัตรู = {'หมากดำ' if bot_player == 1 else 'หมากขาว'}")
    print(f"จำกัดหมากต่อฝ่าย = {stone_limit}")


def is_on_board(x, y):
    return 0 <= x < SIZE and 0 <= y < SIZE


def get_neighbors(x, y):
    neighbors = []
    for dx, dy in [(1,0), (-1,0), (0,1), (0,-1)]:
        nx, ny = x + dx, y + dy
        if is_on_board(nx, ny):
            neighbors.append((nx, ny))
    return neighbors


def switch_player():
    global current_player
    current_player = 2 if current_player == 1 else 1


def player_name(player):
    return "หมากดำ" if player == 1 else "หมากขาว"


def opponent_of(player):
    return 2 if player == 1 else 1



# ฝั่งของหมาก / คำนวนพื้นที่ / การล้อม
def get_group(x, y):
    color = board[y][x]
    if color == 0:
        return set()

    visited = set()
    stack = [(x, y)]

    while stack:
        cx, cy = stack.pop()
        if (cx, cy) in visited:
            continue

        visited.add((cx, cy))

        for nx, ny in get_neighbors(cx, cy):
            if board[ny][nx] == color and (nx, ny) not in visited:
                stack.append((nx, ny))

    return visited


def get_liberties(group):
    liberties = set()

    for x, y in group:
        for nx, ny in get_neighbors(x, y):
            if board[ny][nx] == 0:
                liberties.add((nx, ny))

    return liberties


def remove_group(group):
    for x, y in group:
        board[y][x] = 0


def count_stones(color):
    total = 0
    for row in board:
        total += row.count(color)
    return total

#ส่วนวางหมาก / จับกิน / กันหักขา / ส่วนการคิดของบอ
def place_stone(x, y, player, silent=False, simulate=False):
    global black_captures, white_captures
    global black_used_stones, white_used_stones, stone_limit

    if not is_on_board(x, y):
        if not silent:
            print("เล็งดีๆ ตำแหน่งมันเกินกระดานครับโบล้")
        return False, 0

    if board[y][x] != 0:
        if not silent:
            print("ช่องนี้มีหมากอยู่แล้ว")
        return False, 0

    # เช็คจำนวนหมากก่อนวาง
    if player == 1 and black_used_stones >= stone_limit:
        if not silent:
            print("หมากฝั่งสีดำหมดแล้ว")
        return False, 0

    if player == 2 and white_used_stones >= stone_limit:
        if not silent:
            print("หมากฝั่งสีขาวหมดแล้ว")
        return False, 0

    opponent = opponent_of(player)

    # ส่วนที่บอทใช้วางหมากชั่วคราว (จินตนาการว่าจะลงตรงไหนดีอะ)
    board[y][x] = player

    captured_any = False
    captured_count = 0
    checked_enemy_groups = []

    for nx, ny in get_neighbors(x, y):
        if board[ny][nx] == opponent:
            enemy_group = get_group(nx, ny)

            if enemy_group in checked_enemy_groups:
                continue
            checked_enemy_groups.append(enemy_group)

            enemy_liberties = get_liberties(enemy_group)
            if len(enemy_liberties) == 0:
                captured_count += len(enemy_group)
                remove_group(enemy_group)
                captured_any = True

    my_group = get_group(x, y)
    my_liberties = get_liberties(my_group)

    if len(my_liberties) == 0 and not captured_any:
        board[y][x] = 0
        if not silent:
            print("ห้ามลงแบบฆ่าตัวตาย / ห้ามหักขาตัวเองนะจ้ะ")
        return False, 0

    #ส่วนที่ลดหมากที่ลงเฉพาะตอนลงบอทจริงเท่านั้น
    if not simulate:
        if player == 1:
            black_used_stones += 1
            black_captures += captured_count
        else:
            white_used_stones += 1
            white_captures += captured_count

    return True, captured_count


# ส่วนกระดาานสำรอง แล้วก็คืนค่ากระดาน เอาไว้ใช้ให้บอททดลองเดินก่อนลงจริง
def copy_board_state():
    return [row[:] for row in board]


def restore_board_state(saved):
    global board
    board = [row[:] for row in saved]


def is_valid_move(x, y, player):
    saved = copy_board_state()
    ok, _ = place_stone(x, y, player, silent=True, simulate=True)
    restore_board_state(saved)
    return ok

def get_valid_moves(player):
    moves = []
    for y in range(SIZE):
        for x in range(SIZE):
            if board[y][x] == 0 and is_valid_move(x, y, player):
                moves.append((x, y))
    return moves



# heuristic bot นิสัยของ(Easy Mode)
#-วางตรงที่ได้คะแนนเยอะ   - กลุ่มตัวเองมีพื้นที่มากขึ้นคือได้คะแนนเยอะ
#- จะวางใกล้ๆ พวกเดียวกัน - ชอบหนีศัตรูแบบห่างๆ
#- ส่วนใหญ่จะชอบเลือกกลางกระดาน
def evaluate_move(x, y, player):
    opponent = opponent_of(player)

    saved = copy_board_state()

    before_opponent_stones = count_stones(opponent)

    ok, _ = place_stone(x, y, player, silent=True, simulate=True)
    if not ok:
        restore_board_state(saved)
        return -10**9

    score = 0

    after_opponent_stones = count_stones(opponent)
    captured = before_opponent_stones - after_opponent_stones
    score += captured * 100

    my_group = get_group(x, y)
    my_libs = len(get_liberties(my_group))
    score += my_libs * 8

    friendly_neighbors = 0
    enemy_neighbors = 0
    for nx, ny in get_neighbors(x, y):
        if board[ny][nx] == player:
            friendly_neighbors += 1
        elif board[ny][nx] == opponent:
            enemy_neighbors += 1

    score += friendly_neighbors * 12
    score += enemy_neighbors * 6

    if my_libs == 1:
        score -= 40
    elif my_libs == 2:
        score -= 10

    center = SIZE // 2
    dist = abs(x - center) + abs(y - center)
    score += max(0, 10 - dist)

    score += random.uniform(0, 0.5)

    restore_board_state(saved)
    return score


def bot_heuristic_move(player):
    moves = get_valid_moves(player)
    if not moves:
        return None

    best_score = -10**9
    best_move = None

    for x, y in moves:
        score = evaluate_move(x, y, player)
        if score > best_score:
            best_score = score
            best_move = (x, y)

    return best_move



# ส่วนวาดกระดาน
def draw_board():
    fig_size = 4.0 if SIZE == 9 else 4.6 if SIZE == 13 else 5.2
    stone_size = 180 if SIZE == 9 else 140 if SIZE == 13 else 110
    tick_size = 10 if SIZE == 9 else 9 if SIZE == 13 else 8

    fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    # พื้นหลังไม้โง่ๆ
    ax.set_facecolor("#DEB887")

    # วาดเส้นกระดาน
    for i in range(SIZE):
        ax.plot([0, SIZE - 1], [i, i], color='black', linewidth=0.8)
        ax.plot([i, i], [0, SIZE - 1], color='black', linewidth=0.8)

    # จุดกึ่งกลางตามขนาดแต่ละกระดาน
    if SIZE == 19:
        hoshi_points = [3, 9, 15]
    elif SIZE == 13:
        hoshi_points = [3, 6, 9]
    elif SIZE == 9:
        hoshi_points = [2, 4, 6]
    else:
        hoshi_points = []

    for hx in hoshi_points:
        for hy in hoshi_points:
            ax.scatter(hx, hy, s=10, color='black')

    # วาดหมาก
    for y in range(SIZE):
        for x in range(SIZE):
            if board[y][x] == 1:
                ax.scatter(x, y, s=stone_size, color='black')
            elif board[y][x] == 2:
                ax.scatter(x, y, s=stone_size, color='white', edgecolors='black', linewidths=1)

    # เส้นขอบกระดาน
    ax.set_xlim(-0.7, SIZE - 0.3)
    ax.set_ylim(SIZE - 0.3, -0.7)
    ax.set_aspect('equal')

    # แสดงเลขขอบกระดานให้ดูอะ
    ax.set_xticks(range(SIZE))
    ax.set_yticks(range(SIZE))
    ax.tick_params(axis='both', labelsize=tick_size)

    plt.tight_layout()
    plt.show()

def get_stone_limit(size):
  if size == 9:
      return 40
  elif size == 13:
      return 84
  elif size == 19:
      return 181
  else:
      return (size * size) // 2



# ระบบการเล่น
def bot_play():
    global current_player

    if current_player != bot_player:
        print("ตอนนี้เป็นตาของผู้เล่น")
        return

    move = bot_heuristic_move(bot_player)

    if move is None:
        print("ศัตรูไม่มีหมากแล้ว")
        return

    x, y = move
    ok, captured = place_stone(x, y, bot_player)

    if ok:
        print(f"ศัตรู ({player_name(bot_player)}) ลงที่ ({x}, {y})", end="")
        if captured > 0:
            print(f" และล้อมคุณได้ {captured} เม็ด")
        else:
            print()
        switch_player()
        draw_board()


def human_play(x, y):
    global current_player

    if current_player != human_player:
        print("ตอนนี้ตาของศัตรู")
        return

    ok, captured = place_stone(x, y, human_player)

    if ok:
        print(f"คุณ ({player_name(human_player)}) ลงที่ ({x}, {y})", end="")
        if captured > 0:
            print(f" และล้อมศัตรูได้ {captured} เม็ด")
        else:
            print()
        switch_player()
        draw_board()
    else:
        print("ลงตานี้ไม่ได้")


def play_turn(x, y): #เมื่อคนลงไปแล้ว 1 ตา ถ้าลงได้ บอทจะลงสวนทันที
    global current_player

    # ส่วนของฝั่งผู้ที่เล่นลง
    if current_player == human_player:
        ok, captured = place_stone(x, y, human_player)

        if ok:
            print(f"คุณ ({player_name(human_player)}) ลงที่ ({x}, {y})", end="")
            if captured > 0:
                print(f" และล้อมศัตรูได้ {captured} เม็ด")
            else:
                print()
            switch_player()
            draw_board()
        else:
            print("ลงตานี้ไม่ได้")
            return
    else:
        print("ตอนนี้ตาของศัตรู")
        return

    # ส่วนของฝั่งบอทตอบ
    if current_player == bot_player:
        move = bot_heuristic_move(bot_player)
        if move is None:
            print("ศัตรูไม่มีหมากแล้ว")
            return

        bx, by = move
        ok, captured = place_stone(bx, by, bot_player)

        if ok:
            print(f"ศัตรู ({player_name(bot_player)}) ลงที่ ({bx}, {by})", end="")
            if captured > 0:
                print(f" และล้อมได้ {captured} เม็ด")
            else:
                print()
            switch_player()
            draw_board()



# ตัวช่วยสำหรับดูข้อมูลต่างๆ เช่นช่องนี้ลงได้มั้ย, หมากในช่องนั้นเป็นของฝั่งอะไร
def show_valid_moves(player=None, limit=30):
    if player is None:
        player = current_player

    moves = get_valid_moves(player)
    print(f"จำนวนหมากที่ลงได้ของ {player_name(player)} = {len(moves)}")
    print("ตัวอย่าง:", moves[:limit])


def show_group_and_liberties(x, y):
    if not is_on_board(x, y):
        print("ตำแหน่งอยู่นอกกระดาน")
        return

    if board[y][x] == 0:
        print("ช่องนี้ว่าง")
        return

    group = get_group(x, y)
    liberties = get_liberties(group)

    print(f"หมากที่ ({x}, {y}) เป็นของ {player_name(board[y][x])}")
    print("group =", group)
    print("liberties =", liberties)
    print("จำนวน liberties =", len(liberties))


# ส่วนของคะแนน / กลุ่มที่โดนล้อม / ระบบจบเกม
def get_group_on_temp(temp_board, x, y):
    color = temp_board[y][x]
    if color == 0:
        return set()

    visited = set()
    stack = [(x, y)]

    while stack:
        cx, cy = stack.pop()
        if (cx, cy) in visited:
            continue

        visited.add((cx, cy))

        for nx, ny in get_neighbors(cx, cy):
            if temp_board[ny][nx] == color and (nx, ny) not in visited:
                stack.append((nx, ny))

    return visited


def get_liberties_on_temp(temp_board, group):
    liberties = set()

    for x, y in group:
        for nx, ny in get_neighbors(x, y):
            if temp_board[ny][nx] == 0:
                liberties.add((nx, ny))

    return liberties


def count_eyes_for_group(temp_board, group):#นับ eye condition เมื่อมีหมากสีเดียวกันจับกลุ่มเยอะๆโดยไม่มีหมากศัตรูจะนับว่าสามารถทำ eye condition ได้
    first_x, first_y = next(iter(group))
    color = temp_board[first_y][first_x]

    visited_empty = set()
    eye_count = 0

    candidate_empties = set()
    for x, y in group:
        for nx, ny in get_neighbors(x, y):
            if temp_board[ny][nx] == 0:
                candidate_empties.add((nx, ny))

    for sx, sy in candidate_empties:
        if (sx, sy) in visited_empty:
            continue

        queue = [(sx, sy)]
        region = set()
        border_colors = set()

        while queue:
            cx, cy = queue.pop()
            if (cx, cy) in visited_empty:
                continue
            if temp_board[cy][cx] != 0:
                continue

            visited_empty.add((cx, cy))
            region.add((cx, cy))

            for nx, ny in get_neighbors(cx, cy):
                if temp_board[ny][nx] == 0:
                    if (nx, ny) not in visited_empty:
                        queue.append((nx, ny))
                else:
                    border_colors.add(temp_board[ny][nx])

        if border_colors == {color}:
            eye_count += 1

    return eye_count


def is_dead_group(temp_board, group):#กันลงจุดตาย แต่ถ้าในวงสามารถทำ 2 eyesได้ก็ลงต่อ
    libs = len(get_liberties_on_temp(temp_board, group))
    eyes = count_eyes_for_group(temp_board, group)

    if eyes >= 2:
        return False

    if libs <= 1:
        return True

    if libs <= 2 and eyes == 0:
        return True

    return False


def find_dead_groups(temp_board):
    visited = set()
    dead_black = []
    dead_white = []

    for y in range(SIZE):
        for x in range(SIZE):
            if temp_board[y][x] == 0:
                continue
            if (x, y) in visited:
                continue

            group = get_group_on_temp(temp_board, x, y)
            visited |= group

            if is_dead_group(temp_board, group):
                color = temp_board[y][x]
                if color == 1:
                    dead_black.append(group)
                else:
                    dead_white.append(group)

    return dead_black, dead_white


def remove_dead_groups_from_temp(temp_board, dead_groups):
    for group in dead_groups:
        for x, y in group:
            temp_board[y][x] = 0


def count_territory(temp_board):
    visited = set()
    black_territory = 0
    white_territory = 0

    for y in range(SIZE):
        for x in range(SIZE):
            if temp_board[y][x] != 0 or (x, y) in visited:
                continue

            queue = [(x, y)]
            region = set()
            border_colors = set()

            while queue:
                cx, cy = queue.pop()
                if (cx, cy) in visited:
                    continue
                if temp_board[cy][cx] != 0:
                    continue

                visited.add((cx, cy))
                region.add((cx, cy))

                for nx, ny in get_neighbors(cx, cy):
                    if temp_board[ny][nx] == 0:
                        if (nx, ny) not in visited:
                            queue.append((nx, ny))
                    else:
                        border_colors.add(temp_board[ny][nx])

            if border_colors == {1}:
                black_territory += len(region)
            elif border_colors == {2}:
                white_territory += len(region)

    return black_territory, white_territory

def get_remaining_stones():
    black_remaining = stone_limit - black_used_stones
    white_remaining = stone_limit - white_used_stones
    return black_remaining, white_remaining

def calculate_score(komi=KOMI): #คะแนนแต้มต่อของฝั่งขาว
    temp_board = [row[:] for row in board]

    dead_black, dead_white = find_dead_groups(temp_board)

    dead_black_count = sum(len(g) for g in dead_black)
    dead_white_count = sum(len(g) for g in dead_white)

    remove_dead_groups_from_temp(temp_board, dead_black)
    remove_dead_groups_from_temp(temp_board, dead_white)

    black_territory, white_territory = count_territory(temp_board)

    black_score = black_territory + black_captures + dead_white_count
    white_score = white_territory + white_captures + dead_black_count + komi

    result = {
        "black_territory": black_territory,
        "white_territory": white_territory,
        "black_captures": black_captures,
        "white_captures": white_captures,
        "dead_black_count": dead_black_count,
        "dead_white_count": dead_white_count,
        "black_score": black_score,
        "white_score": white_score,
        "winner": "ดำ" if black_score > white_score else "ขาว" if white_score > black_score else "เสมอ",
        "dead_black_groups": dead_black,
        "dead_white_groups": dead_white,
        "komi": komi,
    }

    return result

def show_score(komi=KOMI):
    result = calculate_score(komi=komi)

    print("========== SCORE ==========")
    print(f"พื้นที่ของหมากดำ   : {result['black_territory']}")
    print(f"พื้นที่ของหมากขาว  : {result['white_territory']}")
    print(f"โดนล้อมโดยหมากดำ    : {result['black_captures']}")
    print(f"โดนล้อมโดยหมากขาว   : {result['white_captures']}")
    print(f"หมากดำที่เป็นเชลย   : {result['dead_black_count']}")
    print(f"หมากขาวที่เป็นเชลย  : {result['dead_white_count']}")
    print(f"Komi ขาว       : {result['komi']}")
    print("---------------------------")
    print(f"คะแนนหมากดำ        : {result['black_score']}")
    print(f"คะแนนหมากขาว       : {result['white_score']}")
    print(f"ผู้ชนะ          : {result['winner']}")
    print("---------------------------")
    print(f"ฝั่งดำใช้หมากไป      : {black_used_stones}/{stone_limit}")
    print(f"ฝั่งขาวใช้หมากไป     : {white_used_stones}/{stone_limit}")
    print("---------------------------")
    print(f"หมากดำเหลือ        : {stone_limit - black_used_stones}")
    print(f"หมากขาวเหลือ       : {stone_limit - white_used_stones}")

    return result

def show_stone_status():
    print("===== STONE STATUS =====")
    print(f"จำกัดหมาก 2 ฝั่งไว้ที่ : {stone_limit}")
    print(f"หมากดำใช้ไป       : {black_used_stones}")
    print(f"หมากขาวใช้ไป      : {white_used_stones}")
    print("---------------------------")


def show_remaining_stones():
    black_remaining, white_remaining = get_remaining_stones()
    print("===== REMAINING STONES =====")
    print(f"หมากดำเหลือ  : {black_remaining}")
    print(f"หมากขาวเหลือ : {white_remaining}")
    print("---------------------------")

def end_game(komi=KOMI):
    global game_over
    game_over = True

    print("จบเกม กำลังคำนวณคะแนนสักครู่")
    result = show_score(komi=komi)
    return result




In [171]:
# MCTS-lite Normal mode
# - นิสัยการเล่น คล้ายๆ Easy mode แต่จะคัดละเอียดกว่า
# - ยังมีนิสัยชอบวางตรงกลางเหมือนเดิม -ล้อมได้จะล้อมเลย เพราะการคัดคะแนนจะเอา 5-8 อันดับ ของ Easy mode ที่มีคะแนนเยอะสุดมาแล้วคัดอีกที
def snapshot_game():
    return {
        "board": [row[:] for row in board],
        "current_player": current_player,
        "black_captures": black_captures,
        "white_captures": white_captures,
        "black_used_stones": black_used_stones,
        "white_used_stones": white_used_stones,
        "stone_limit": stone_limit,
        "consecutive_passes": consecutive_passes,
        "game_over": game_over,
        "SIZE": SIZE,
        "human_player": human_player,
        "bot_player": bot_player,
    }

def restore_game(state):
    global board, current_player
    global black_captures, white_captures
    global black_used_stones, white_used_stones, stone_limit
    global consecutive_passes, game_over
    global SIZE, human_player, bot_player

    board = [row[:] for row in state["board"]]
    current_player = state["current_player"]
    black_captures = state["black_captures"]
    white_captures = state["white_captures"]
    black_used_stones = state["black_used_stones"]
    white_used_stones = state["white_used_stones"]
    stone_limit = state["stone_limit"]
    consecutive_passes = state["consecutive_passes"]
    game_over = state["game_over"]
    SIZE = state["SIZE"]
    human_player = state["human_player"]
    bot_player = state["bot_player"]

def get_score_winner():
    result = calculate_score(komi=KOMI)
    if result["black_score"] > result["white_score"]:
        return 1
    elif result["white_score"] > result["black_score"]:
        return 2
    return 0

def random_playout(start_player, max_steps=25):
    player = start_player
    passes = 0

    for _ in range(max_steps):
        moves = get_valid_moves(player)

        if not moves:
            passes += 1
            if passes >= 2:
                break
            player = opponent_of(player)
            continue

        x, y = random.choice(moves)
        ok, _ = place_stone(x, y, player, silent=True, simulate=False)

        if ok:
            passes = 0
        else:
            passes += 1

        if passes >= 2:
            break

        player = opponent_of(player)

    return get_score_winner()

def mcts_move(player, candidate_limit=10, simulations_per_move=2, max_playout_steps=25):
    moves = get_valid_moves(player)
    if not moves:
        return None

    # ตรงนี้ให้คัด move ที่ดูดีด้วย heuristic ก่อน ไม่งั้นจะทำงานช้ามากช่วยลดเวลาทำงาน
    scored_moves = []
    for x, y in moves:
        score = evaluate_move(x, y, player)
        scored_moves.append((score, (x, y)))

    scored_moves.sort(reverse=True, key=lambda item: item[0])
    candidates = [move for score, move in scored_moves[:candidate_limit]]

    root_state = snapshot_game()
    best_move = None
    best_score = -1

    for move in candidates:
        wins = 0
        draws = 0

        for _ in range(simulations_per_move):
            restore_game(root_state)

            ok, _ = place_stone(move[0], move[1], player, silent=True, simulate=False)
            if not ok:
                continue

            winner = random_playout(opponent_of(player), max_steps=max_playout_steps)

            if winner == player:
                wins += 1
            elif winner == 0:
                draws += 1

        score = wins + draws * 0.5

        if score > best_score:
            best_score = score
            best_move = move

    restore_game(root_state)

    if best_move is None:
        return bot_heuristic_move(player)

    return best_move

def choose_bot_move():
    if BOT_MODE == "heuristic":
        return bot_heuristic_move(bot_player)

    if BOT_MODE == "mcts":
        try:
            if SIZE == 9:
                return mcts_move(bot_player, candidate_limit=10, simulations_per_move=2, max_playout_steps=25)
            elif SIZE == 13:
                return mcts_move(bot_player, candidate_limit=8, simulations_per_move=1, max_playout_steps=20)
            else:
                return mcts_move(bot_player, candidate_limit=6, simulations_per_move=1, max_playout_steps=15)
        except Exception as e:
            print("Nomal mode error:", e)
            print("Use easy mode")
            return bot_heuristic_move(bot_player)

    return bot_heuristic_move(bot_player)

In [172]:
# ตัวแปร global ไว้บอกว่าเลือกศัตรูแบบไหน
BOT_MODE = "heuristic"

board_output = widgets.Output()
control_output = widgets.Output()

# ส่วนของ UI
bot_selector = widgets.Dropdown(
    options=[
        ("Easy mode", "heuristic"),
        ("Normal mode", "mcts"),
    ],
    description="Bot:"
)

size_selector = widgets.Dropdown(
    options=[
        ("9x9", 9),
        ("13x13", 13),
        ("19x19", 19),
    ],
    value=19,
    description="Board:"
)

side_selector = widgets.Dropdown(
    options=[
        ("หมากดำ", 1),
        ("หมากขาว", 2),
    ],
    value=1,
    description="Side:"
)

start_button = widgets.Button(description="เริ่มเกม", button_style='success')
move_input = widgets.Text(placeholder="3,4", description="วางที่ตำแหน่ง:")
move_button = widgets.Button(description="ลงหมาก", button_style="primary")
pass_button = widgets.Button(description="ผ่านเทริน", button_style="warning")
end_button = widgets.Button(description="จบเกมทันที", button_style="danger")


# ================= UI STATE =================
def hide_settings():
    # ซ่อน setting เกมระหว่างเล่น
    bot_selector.layout.display = "none"
    size_selector.layout.display = "none"
    side_selector.layout.display = "none"
    start_button.layout.display = "none"

    # เปิดปุ่มพวก ลงหมาก,ผ่านเทริน,จบเกม
    move_input.disabled = False
    move_button.disabled = False
    pass_button.disabled = False
    end_button.disabled = False


def show_settings():
    # เอา setting กลับมาตอนเกมจบ
    bot_selector.layout.display = ""
    size_selector.layout.display = ""
    side_selector.layout.display = ""
    start_button.layout.display = ""

    # ปิดปุ่มโซนการเล่นจนกว่าจะเริ่มเกมใหม่
    move_input.disabled = True
    move_button.disabled = True
    pass_button.disabled = True
    end_button.disabled = True


# โชว์ setting แต่จะปิดไม่ให้เล่นจนกว่าจะเริ่มเกมนะจ้ะ
show_settings()


# ระบบต่างๆ ของปุ่มการส่งค่า return หาฟังก์ชั่นต่างๆ
def redraw_board():
    with board_output:
        clear_output(wait=True)
        draw_board()


def handle_pass(player):
    global consecutive_passes

    consecutive_passes += 1
    print(f"{player_name(player)} ขอผ่าน")

    if consecutive_passes >= 2:
        return "end"

    switch_player()
    return "pass"


def finish_game_by_pass():
    print("ผู้เล่นทั้ง 2 ฝั่งขอผ่านดังนั้นจบเกม")
    end_game()
    show_settings()


def on_start_clicked(b):
    global BOT_MODE

    BOT_MODE = bot_selector.value
    selected_size = size_selector.value
    selected_side = side_selector.value

    with control_output:
        clear_output(wait=True)
        reset_game(size=selected_size, human=selected_side)

        print(f"เริ่มเกม: {selected_size}x{selected_size} | bot = {BOT_MODE}")
        print(f"หมากของผู้เล่น = {'ดำ' if selected_side == 1 else 'ขาว'}")

    hide_settings()
    redraw_board()

    # ถ้าผู้เล่นเลือกขาว ศัตรูต้องไปดำแล้ววางหมากก่อน
    if human_player == 2:
        move = choose_bot_move()

        with control_output:
            clear_output(wait=True)
            print(f"เริ่มเกม: {selected_size}x{selected_size} | bot = {BOT_MODE}")
            print("หมากของผู้เล่น = ขาว")

            if move:
                bx, by = move
                ok, _ = place_stone(bx, by, bot_player)

                if ok:
                    switch_player()
                    redraw_board()

                    black_remaining, white_remaining = get_remaining_stones()
                    print(f"ศัตรูเลือกเปิดที่ ({bx},{by})")
                    print(f"หมากดำเหลือ {black_remaining} | หมากขาวเหลือ {white_remaining}")
                else:
                    print("ศัตรูเปิดเกมไม่ได้")
            else:
                print("ศัตรูไม่สามารถลงได้ขอผ่าน")
                result = handle_pass(bot_player)

                if result == "end":
                    finish_game_by_pass()


def on_move_clicked(b):
    global consecutive_passes

    with control_output:
        clear_output(wait=True)

        try:
            x, y = map(int, move_input.value.split(","))
            ok, captured = place_stone(x, y, human_player)

            if ok:
                consecutive_passes = 0
                switch_player()
                redraw_board()

                print(f"คุณลงที่ ({x},{y})")
                if captured > 0:
                    print(f"คุณล้อมศัตรูได้ {captured} เม็ด")

                move = choose_bot_move()

                if move:
                    bx, by = move
                    ok2, captured2 = place_stone(bx, by, bot_player)

                    if ok2:
                        consecutive_passes = 0
                        switch_player()
                        redraw_board()

                        black_remaining, white_remaining = get_remaining_stones()
                        print(f"ศัตรูเลือกลงที่ ({bx},{by})")
                        if captured2 > 0:
                            print(f"ศัตรูล้อมหมากคุณได้ {captured2} เม็ด")
                        print(f"หมากดำเหลือ {black_remaining} | หมากขาวเหลือ {white_remaining}")
                    else:
                        print("ศัตรูลงไม่ได้")
                else:
                    print("ศัตรูไม่สามารถลงได้ขอผ่าน")
                    result = handle_pass(bot_player)

                    if result == "end":
                        finish_game_by_pass()
                        return
            else:
                print("ลงไม่ได้")

            move_input.value = ""

        except Exception as e:
            print("กรอกตำแหน่งเช่น 3,4")
            # ถ้าอยาก debug บรรทัดเนี้ย พิมพ์ print(e) หนา

def on_pass_clicked(b):
    global consecutive_passes

    with control_output:
        clear_output(wait=True)

        # ผู้เล่นขอผ่านเทริน
        result = handle_pass(human_player)

        if result == "end":
            finish_game_by_pass()
            return

        redraw_board()

        # เทรินศัตรู
        move = choose_bot_move()

        if move:
            bx, by = move
            ok, captured = place_stone(bx, by, bot_player)

            if ok:
                consecutive_passes = 0
                switch_player()
                redraw_board()

                black_remaining, white_remaining = get_remaining_stones()
                print(f"ศัตรูเลือกลงที่ ({bx},{by})")
                if captured > 0:
                    print(f"ศัตรูล้อมหมากคุณได้ {captured} เม็ด")
                print(f"หมากดำเหลือ {black_remaining} | หมากขาวเหลือ {white_remaining}")
            else:
                print("ศัตรูลงไม่ได้")
        else:
            print("ศัตรูไม่สามารถลงได้ขอผ่าน")
            result = handle_pass(bot_player)

            if result == "end":
                finish_game_by_pass()
                return


def on_end_game_clicked(b):
    with control_output:
        clear_output(wait=True)
        end_game()

    show_settings()


# bind
start_button.on_click(on_start_clicked)
move_button.on_click(on_move_clicked)
pass_button.on_click(on_pass_clicked)
end_button.on_click(on_end_game_clicked)


# การแสดงปุ่ม การจัดการเรียกใช้ระบบต่างๆ (Layout)
control_panel = widgets.VBox([
    bot_selector,
    size_selector,
    side_selector,
    start_button,
    move_input,
    move_button,
    pass_button,
    end_button,
    control_output
])

app = widgets.VBox([
    board_output,
    control_panel
])

display(app)